# Data Cleaning & Preparação

## Objetivo

Aplicar tratamento estruturado aos problemas identificados na etapa de auditoria, garantindo consistência semântica, tipagem correta e integridade das regras de negócio.

Todos os ajustes realizados serão documentados e justificados.

---
# 01 - Imports

In [1]:
import pandas as pd
import numpy as np

In [27]:
df = pd.read_csv("../data/raw/vendas_loja_expandida.csv")

In [28]:
df.head()

,ID da Venda,Data da Venda,Produto,Categoria,Preço Unitário,Quantidade Vendida,Desconto,Valor Total,Localização,Forma de Pagamento
0,1001,2023-01-01 00:00:00.000000000,Livro G,Livros,60.00,6,0.05,57.00,Curitiba,Pix
1,1002,2023-01-01 05:49:40.386924616,Camisa D,Roupas,80.00,4,0.04,307.20,Salvador,Cartão de Débito
2,1003,2023-01-01 11:39:20.773849232,Calça E,Roupas,120.00,6,0.02,705.60,Curitiba,Boleto
3,1004,2023-01-01 17:29:01.160773849,Livro G,Livros,60.00,2,0.02,117.60,Salvador,Pix
4,1005,2023-01-01 23:18:41.547698465,Fone C,Eletrônicos,1626.54,2,0.07,3025.36,São Paulo,Cartão de Débito


# 02 - Padronização e Tipagem das Variáveis

Nesta etapa serão aplicados ajustes estruturais nas variáveis identificadas na auditoria, incluindo:

- Conversão de datas
- Padronização de campos categóricos
- Correção de tipagem numérica
- Normalização textual

## 02.1 - Conversão de Datas

In [29]:
df["Data da Venda"] = pd.to_datetime(
    df["Data da Venda"],
    format="mixed"
)

In [30]:
df["Data da Venda"].min(), df["Data da Venda"].max()

(Timestamp('2023-01-01 00:00:00'), Timestamp('2026-01-15 10:00:00'))

---
**Validação da Conversão de Datas**

A coluna "Data da Venda" foi convertida para datetime utilizando `pd.to_datetime()` com `format="mixed"`.

Resultado:

- 0 valores nulos após conversão
- Intervalo temporal consistente
- Nenhuma perda de informação

Conclusão:
A variável está íntegra e pronta para derivação de features temporais.

## 02.2 - Padronização de Variáveis Categóricas

### 02.2.1 - Categoria

In [31]:
df["Categoria"] = df["Categoria"].str.strip()

In [32]:
df["Categoria"].nunique()

7

---
**Resultado da Padronização - Categoria**

A variável foi normalizada via remoção de whitespace.

Resultado:
- Redução de 14 para 7 categorias únicas.
- Eliminação de fragmentação textual.
- Domínio categórico consolidado.

A variável está consistente para agregações analíticas.

### 02.2.1 - Forma de Pagamento

In [33]:
df["Forma de Pagamento"].value_counts()

Forma de Pagamento
Cartão de Crédito    3470
Pix                  3231
Cartão de Débito     3180
Boleto               2951
Cartao de Credito      62
Credito                48
boleto                 43
Cartão credito         38
pix                    36
debito                 35
Name: count, dtype: int64

In [34]:
df["Forma de Pagamento"] = (
    df["Forma de Pagamento"]
    .str.strip()
    .str.lower()
)

In [35]:
df["Forma de Pagamento"].value_counts()

Forma de Pagamento
cartão de crédito    3470
pix                  3267
cartão de débito     3180
boleto               2994
cartao de credito      62
credito                48
cartão credito         38
debito                 35
Name: count, dtype: int64

In [36]:
map_pagamento = {
    "cartao de credito": "Cartão de Crédito",
    "cartão de crédito": "Cartão de Crédito",
    "cartão credito": "Cartão de Crédito",
    "credito": "Cartão de Crédito",
    "cartão de débito": "Cartão de Débito",
    "debito": "Cartão de Débito",
    "pix": "Pix",
    "boleto": "Boleto"
}

df["Forma de Pagamento"] = df["Forma de Pagamento"].map(map_pagamento)

In [37]:
df["Forma de Pagamento"].value_counts(dropna = False)

Forma de Pagamento
Cartão de Crédito    3618
Pix                  3267
Cartão de Débito     3215
Boleto               2994
Name: count, dtype: int64

In [38]:
df["Forma de Pagamento"].nunique()

4

---
**Resultado da Padronização - Forma de Pagamento**

A variável foi normalizada via:

- Remoção de whitespace
- Padronização para minúsculas
- Mapeamento semântico

Resultado:
- Consolidação para 4 categorias válidas
- Eliminação de fragmentação textual (~2% da base)
- Domínio consistente para análise financeira

### 02.2.3 - Localização

In [39]:
df["Localização"] = (
    df["Localização"]
    .str.strip()
    .str.lower()
    .str.replace("@", "a", regex=False)
    .str.title()
)

In [40]:
df["Localização"].nunique()

12

In [41]:
df["Localização"].value_counts(dropna=False)

Localização
São Paulo         1506
Rio De Janeiro    1348
Belo Horizonte    1196
Curitiba          1166
Salvador          1150
Goiânia            989
Recife             975
Porto Alegre       969
Brasília           960
Campinas           949
Fortaleza          944
Florianópolis      942
Name: count, dtype: int64

---
**Resultado da Padronização - Localização**

A variável foi normalizada via:

- Remoção de whitespace
- Padronização de caixa
- Correção de ruído textual (substituição de "@")
- Ajuste de capitalização

Resultado:
- Consolidação de 57 variações aparentes para 12 cidades válidas.
- Eliminação de fragmentação regional.
- Base pronta para análise geográfica.

## 02.3 - Conversão e Padronização do Campo "Desconto"

In [42]:
df["Desconto"] = (
    df["Desconto"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

df.loc[df["Desconto"] > 1, "Desconto"] /= 100

In [43]:
df["Desconto"].min(), df["Desconto"].max()

(np.float64(0.0), np.float64(1.0))

---
**Padronização da Variável "Desconto"**

A variável apresentava formato misto (decimal e percentual como string).

Procedimentos aplicados:
- Remoção do símbolo "%"
- Conversão para tipo float
- Normalização de valores percentuais (>1) para formato proporcional

Resultado:
- Intervalo final entre 0 e 1
- Tipagem numérica consistente
- Variável apta para cálculo financeiro

## 02.4 - Consistência de Preço por Produto

In [61]:
preco_real = (
    df[df["Preço Unitário"] > 0]
    .drop_duplicates(subset=["Produto", "Preço Unitário"])
    .set_index("Produto")["Preço Unitário"]
)

In [64]:
df.loc[df["Preço Unitário"] == 0, "Preço Unitário"] = (
    df.loc[df["Preço Unitário"] == 0, "Produto"]
    .map(preco_real)
)

In [66]:
(df["Preço Unitário"] == 0).sum()

np.int64(0)

In [68]:
df.groupby("Produto")["Preço Unitário"].nunique().value_counts()

Preço Unitário
1    49
Name: count, dtype: int64

---
**Tratamento de Preço Unitário Zero**

Identificado erro estrutural onde 46 produtos possuíam registros com preço unitário igual a zero.

Como cada produto possuía um único preço válido (>0), os valores zero foram substituídos pelo preço real correspondente ao produto.

A imputação foi determinística e baseada na própria base de dados.

## 02.5 - Tratamento do Identificador de Venda

In [69]:
df["id_registro"] = range(1, len(df) + 1)

---
**Resultado do Tratamento do Identificador**

O campo "ID da Venda" foi mantido na base para fins de rastreabilidade histórica.

Entretanto, devido à reutilização indevida identificada na etapa de auditoria, ele não pode ser considerado uma chave primária confiável.

Foi criada a variável `id_registro`, que:

- Garante unicidade para cada linha da base
- Preserva a integridade estrutural do dataset
- Permite utilização segura em operações de junção (joins) e modelagem

Com isso, a base passa a possuir uma chave técnica válida para uso analítico.

## 02.6 - Tratamento da Variável "Quantidade Vendida"

In [77]:
df["flag_quantidade_invalida"] = df["Quantidade Vendida"] <= 0

In [78]:
df_analitico = df[df["Quantidade Vendida"] > 0].copy()

In [79]:
len(df), len(df_analitico)

(13094, 12830)

---
**Tratamento da Variável "Quantidade Vendida"**

### Diagnóstico

A distribuição da variável indicou a presença de valores inconsistentes:

- 133 registros com Quantidade Vendida = 0
- 131 registros com Quantidade Vendida = -1

Total de registros potencialmente inválidos: 264.

### Interpretação

- Quantidade = 0 representa venda sem item, o que é comercialmente incoerente.
- Quantidade = -1 pode representar estorno ou devolução, porém não há variável que confirme esse contexto na base.

### Decisão de Tratamento

Como o objetivo da análise é avaliar desempenho comercial (vendas efetivas), foi adotada a seguinte estratégia:

- Manter todos os registros na base limpa (`df`) para rastreabilidade.
- Criar uma base analítica (`df_analitico`) contendo apenas registros com Quantidade Vendida > 0.

Essa decisão garante coerência nas métricas de volume e faturamento, evitando distorções por registros estruturalmente inválidos.

## 02.7 - Recalculo do Valor Total

In [81]:
df_analitico["Valor_Total_Corrigido"] = (
    df_analitico["Preço Unitário"]
    * df_analitico["Quantidade Vendida"]
    * (1 - df_analitico["Desconto"])
)

In [83]:
df_analitico[["Preço Unitário", "Quantidade Vendida", "Desconto", "Valor Total", "Valor_Total_Corrigido"]].head()

,Preço Unitário,Quantidade Vendida,Desconto,Valor Total,Valor_Total_Corrigido
0,60.00,6,0.05,57.00,342.0000
1,80.00,4,0.04,307.20,307.2000
2,120.00,6,0.02,705.60,705.6000
3,60.00,2,0.02,117.60,117.6000
4,1626.54,2,0.07,3025.36,3025.3644


In [86]:
(df_analitico["Valor Total"] - df_analitico["Valor_Total_Corrigido"]).abs().sum()

np.float64(510928.8219000002)

In [89]:
df_analitico["Valor_Total_Corrigido"].sum()

np.float64(45655928.40810001)

In [88]:
df_analitico["Valor Total"].sum(),

(np.float64(45223958.66),)

---
**Impacto do Recalculo do Valor Total**

Após aplicação da regra de negócio correta, observou-se diferença acumulada de aproximadamente R$ 510 mil no faturamento total.

A base original apresentava subavaliação de ~1,1% da receita.

Conclusão:
O recálculo foi necessário para garantir integridade financeira e confiabilidade das análises posteriores.

# Conclusão da Etapa de Data Cleaning

A base de dados passou por tratamento estruturado com foco em integridade, consistência semântica e aderência às regras de negócio.

Principais ajustes realizados:

- Padronização de categorias, formas de pagamento e localizações
- Conversão adequada de datas e descontos
- Correção de preços unitários inválidos
- Tratamento de quantidades inconsistentes
- Criação de chave técnica única (`id_registro`)
- Recalculo do valor total com base na regra de negócio formal

A base analítica (`df_analitico`) agora representa apenas vendas válidas (Quantidade > 0) e utilizará exclusivamente a variável `Valor_Total_Corrigido` para métricas financeiras.

A etapa de preparação está concluída e a base encontra-se pronta para análise exploratória e geração de insights.

In [90]:
df_analitico.to_csv("../data/processed/vendas_tratadas.csv", index=False)

In [91]:
df.to_csv("../data/processed/vendas_clean_completa.csv", index=False)